# **Bitcoin Price Prediction - Naive Bayes Model**

Implementing a Naive Bayes classifier for Bitcoin trading predictions:

##  1: Setup and Imports

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print(" "*8 + "BITCOIN PREDICTION: MODEL COMPARISON")
print(" "*12 + "Random Forest vs Naive Bayes")
print("="*70)

        BITCOIN PREDICTION: MODEL COMPARISON
            Random Forest vs Naive Bayes


## 1: Stratified Sampling + Class Weights

In [35]:
def create_realistic_target(returns, threshold_percentile=70):
    """
    Create target based on percentile thresholds rather than fixed %.
    This ensures balanced classes regardless of market conditions.
    """
    up_threshold = np.percentile(returns.dropna(), threshold_percentile)
    down_threshold = np.percentile(returns.dropna(), 100 - threshold_percentile)
    
    target = pd.Series(index=returns.index, dtype=int)
    target[returns > up_threshold] = 1  # Top 30% moves = UP
    target[returns < down_threshold] = 0  # Bottom 30% moves = DOWN
    target[(returns >= down_threshold) & (returns <= up_threshold)] = -1  # Middle 40% = NEUTRAL
    
    return target, up_threshold, down_threshold

def select_best_features(X, y, k=15):
    """Select top K features based on mutual information"""
    from sklearn.feature_selection import SelectKBest, mutual_info_classif
    
    # Remove any remaining NaN
    X_clean = X.fillna(X.median())
    
    selector = SelectKBest(mutual_info_classif, k=min(k, X.shape[1]))
    selector.fit(X_clean, y)
    
    scores = pd.DataFrame({
        'feature': X.columns,
        'score': selector.scores_
    }).sort_values('score', ascending=False)
    
    selected = scores.head(k)['feature'].tolist()
    return selected, scores

## 4: Loading dataset

In [36]:
print("\n[1/10] Loading dataset...")
df = pd.read_csv('../data/features/btc_features_complete.csv', index_col=0, parse_dates=True)
print(f"✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")


[1/10] Loading dataset...
✓ Loaded: 51,443 rows × 86 cols


## 5: Creating balanced target using percentile approach

In [37]:
print("\n[2/10] Creating balanced target using percentile approach...")

y_24h, up_thresh, down_thresh = create_realistic_target(
    df['future_return_24h'], 
    threshold_percentile=65  # Top 35% = UP, Bottom 35% = DOWN, Middle 30% = NEUTRAL
)

# Remove neutral samples
valid_mask = y_24h != -1
df_filtered = df[valid_mask].copy()
y_filtered = y_24h[valid_mask].copy()

print(f"✓ Target created with percentile thresholds:")
print(f"  UP threshold:   {up_thresh:.4f} ({up_thresh*100:.2f}%)")
print(f"  DOWN threshold: {down_thresh:.4f} ({down_thresh*100:.2f}%)")
print(f"\n  Total samples after filtering: {len(y_filtered):,}")
print(f"  UP:   {(y_filtered == 1).sum():,} ({(y_filtered == 1).sum()/len(y_filtered)*100:.1f}%)")
print(f"  DOWN: {(y_filtered == 0).sum():,} ({(y_filtered == 0).sum()/len(y_filtered)*100:.1f}%)")



[2/10] Creating balanced target using percentile approach...
✓ Target created with percentile thresholds:
  UP threshold:   0.0082 (0.82%)
  DOWN threshold: -0.0060 (-0.60%)

  Total samples after filtering: 36,010
  UP:   18,005 (50.0%)
  DOWN: 18,005 (50.0%)


## 6: Feature engineering

In [38]:
print("\n[3/10] Feature engineering...")

# Drop targets and raw OHLC
drop_cols = [
    'future_return_1h', 'future_return_6h', 'future_return_24h',
    'target_direction_1h', 'target_multiclass_1h', 'target_return_1h',
    'Close', 'Open', 'High', 'Low', 'fear_greed_classification'
]
X = df_filtered.drop(columns=drop_cols, errors='ignore')
X = X.select_dtypes(include=[np.number])

# Handle inf/nan
X.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"✓ Starting features: {X.shape[1]}")


[3/10] Feature engineering...
✓ Starting features: 75


## 7: Chronological train/val/test split

In [39]:
print("\n[4/10] Chronological train/val/test split...")

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y_filtered.iloc[:train_end].copy()
y_val = y_filtered.iloc[train_end:val_end].copy()
y_test = y_filtered.iloc[val_end:].copy()

print(f"✓ Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"\n  Train class balance:")
print(f"    UP:   {(y_train == 1).sum():,} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")
print(f"    DOWN: {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")



[4/10] Chronological train/val/test split...
✓ Train: 25,207 | Val: 5,401 | Test: 5,402

  Train class balance:
    UP:   12,669 (50.3%)
    DOWN: 12,538 (49.7%)


## 8: Feature selection using mutual information

In [40]:
print("\n[5/10] Feature selection using mutual information...")

# Impute train set
medians = X_train.median()
X_train_clean = X_train.fillna(medians)

# Select top features
selected_features, feature_scores = select_best_features(X_train_clean, y_train, k=20)

print(f"✓ Selected top 20 features:")
for i, feat in enumerate(selected_features[:10], 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:30s} (score: {score:.4f})")
print(f"  ... and 10 more")

X_train = X_train[selected_features].copy()
X_val = X_val[selected_features].copy()
X_test = X_test[selected_features].copy()


[5/10] Feature selection using mutual information...
✓ Selected top 20 features:
   1. nvt_ratio                      (score: 0.3566)
   2. market_price_usd               (score: 0.3150)
   3. tx_fees_btc                    (score: 0.3141)
   4. hash_rate_change_30d           (score: 0.3131)
   5. avg_block_size_mb              (score: 0.3114)
   6. tx_count_change_7d             (score: 0.3108)
   7. tx_count_daily                 (score: 0.3105)
   8. sp500_change_7d                (score: 0.3019)
   9. hash_rate_change_7d            (score: 0.2949)
  10. SP500                          (score: 0.2942)
  ... and 10 more


## 9: Data preprocessing

In [41]:
print("\n[6/10] Data preprocessing...")

# Fill NaN
X_train.fillna(medians[selected_features], inplace=True)
X_val.fillna(medians[selected_features], inplace=True)
X_test.fillna(medians[selected_features], inplace=True)

# Use RobustScaler (less sensitive to outliers than StandardScaler)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled with RobustScaler")


[6/10] Data preprocessing...
✓ Features scaled with RobustScaler


##  10: Training Naive Bayes

In [43]:
print("\n[7/10] Training Naive Bayes (baseline)...")

nb_model = GaussianNB(var_smoothing=1e-9)
nb_model.fit(X_train_scaled, y_train)

y_val_pred_nb = nb_model.predict(X_val_scaled)
y_val_proba_nb = nb_model.predict_proba(X_val_scaled)[:, 1]

nb_acc = accuracy_score(y_val, y_val_pred_nb)
nb_prec = precision_score(y_val, y_val_pred_nb, zero_division=0)
nb_rec = recall_score(y_val, y_val_pred_nb, zero_division=0)
nb_f1 = f1_score(y_val, y_val_pred_nb, zero_division=0)
nb_roc = roc_auc_score(y_val, y_val_proba_nb)

print(f"\n📊 Naive Bayes - Validation:")
print(f"  Accuracy:  {nb_acc:.4f}")
print(f"  Precision: {nb_prec:.4f}")
print(f"  Recall:    {nb_rec:.4f}")
print(f"  F1 Score:  {nb_f1:.4f}")
print(f"  ROC-AUC:   {nb_roc:.4f}")



[7/10] Training Naive Bayes (baseline)...

📊 Naive Bayes - Validation:
  Accuracy:  0.5249
  Precision: 0.5228
  Recall:    0.8860
  F1 Score:  0.6576
  ROC-AUC:   0.5646


## 11: Training Random Forest with class weights

In [44]:
print("\n[8/10] Training Random Forest with class weights...")

# Calculate class weights to handle imbalance
class_counts = np.bincount(y_train)
class_weights = {0: len(y_train) / (2 * class_counts[0]),
                 1: len(y_train) / (2 * class_counts[1])}

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=100,
    min_samples_leaf=50,
    max_features='sqrt',
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

y_val_pred_rf = rf_model.predict(X_val_scaled)
y_val_proba_rf = rf_model.predict_proba(X_val_scaled)[:, 1]

rf_acc = accuracy_score(y_val, y_val_pred_rf)
rf_prec = precision_score(y_val, y_val_pred_rf, zero_division=0)
rf_rec = recall_score(y_val, y_val_pred_rf, zero_division=0)
rf_f1 = f1_score(y_val, y_val_pred_rf, zero_division=0)
rf_roc = roc_auc_score(y_val, y_val_proba_rf)

print(f"\n📊 Random Forest - Validation:")
print(f"  Accuracy:  {rf_acc:.4f}")
print(f"  Precision: {rf_prec:.4f}")
print(f"  Recall:    {rf_rec:.4f}")
print(f"  F1 Score:  {rf_f1:.4f}")
print(f"  ROC-AUC:   {rf_roc:.4f}")


[8/10] Training Random Forest with class weights...

📊 Random Forest - Validation:
  Accuracy:  0.5306
  Precision: 0.5406
  Recall:    0.5894
  F1 Score:  0.5639
  ROC-AUC:   0.5212


##  12: Test set evaluation

In [45]:
print("\n[9/10] Test set evaluation...")

# Naive Bayes on test
y_test_pred_nb = nb_model.predict(X_test_scaled)
y_test_proba_nb = nb_model.predict_proba(X_test_scaled)[:, 1]
nb_test_roc = roc_auc_score(y_test, y_test_proba_nb)

# Random Forest on test
y_test_pred_rf = rf_model.predict(X_test_scaled)
y_test_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
rf_test_roc = roc_auc_score(y_test, y_test_proba_rf)

print(f"\n📊 TEST SET COMPARISON:")
print(f"\n  Naive Bayes:")
print(f"    Accuracy:  {accuracy_score(y_test, y_test_pred_nb):.4f}")
print(f"    Precision: {precision_score(y_test, y_test_pred_nb, zero_division=0):.4f}")
print(f"    Recall:    {recall_score(y_test, y_test_pred_nb, zero_division=0):.4f}")
print(f"    ROC-AUC:   {nb_test_roc:.4f}")

print(f"\n  Random Forest:")
print(f"    Accuracy:  {accuracy_score(y_test, y_test_pred_rf):.4f}")
print(f"    Precision: {precision_score(y_test, y_test_pred_rf, zero_division=0):.4f}")
print(f"    Recall:    {recall_score(y_test, y_test_pred_rf, zero_division=0):.4f}")
print(f"    ROC-AUC:   {rf_test_roc:.4f}")

print(f"\n  🏆 Improvement: {(rf_test_roc - nb_test_roc):.4f} ROC-AUC points")

# Confusion matrices
print(f"\n  Naive Bayes Confusion Matrix:")
cm_nb = confusion_matrix(y_test, y_test_pred_nb)
print(f"              Predicted DOWN | Predicted UP")
print(f"  Actual DOWN: {cm_nb[0][0]:5d}      | {cm_nb[0][1]:5d}")
print(f"  Actual UP:   {cm_nb[1][0]:5d}      | {cm_nb[1][1]:5d}")

print(f"\n  Random Forest Confusion Matrix:")
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
print(f"              Predicted DOWN | Predicted UP")
print(f"  Actual DOWN: {cm_rf[0][0]:5d}      | {cm_rf[0][1]:5d}")
print(f"  Actual UP:   {cm_rf[1][0]:5d}      | {cm_rf[1][1]:5d}")



[9/10] Test set evaluation...

📊 TEST SET COMPARISON:

  Naive Bayes:
    Accuracy:  0.4767
    Precision: 0.4747
    Recall:    0.9977
    ROC-AUC:   0.4945

  Random Forest:
    Accuracy:  0.5020
    Precision: 0.4443
    Recall:    0.2106
    ROC-AUC:   0.4648

  🏆 Improvement: -0.0297 ROC-AUC points

  Naive Bayes Confusion Matrix:
              Predicted DOWN | Predicted UP
  Actual DOWN:    26      |  2821
  Actual UP:       6      |  2549

  Random Forest Confusion Matrix:
              Predicted DOWN | Predicted UP
  Actual DOWN:  2174      |   673
  Actual UP:    2017      |   538


## 13: Feature importance analysis

In [46]:
print("\n[10/10] Feature importance analysis (Random Forest)...")

feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n📊 Top 10 Most Important Features:")
for i, row in feature_importance.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:.4f}")


[10/10] Feature importance analysis (Random Forest)...

📊 Top 10 Most Important Features:
  btc_dxy_correlation            0.0733
  market_price_usd               0.0674
  DXY                            0.0663
  nvt_ratio                      0.0591
  btc_sp500_correlation          0.0557
  sp500_change_7d                0.0557
  btc_gold_correlation           0.0536
  total_btc_supply               0.0531
  tx_count_change_7d             0.0515
  VIX                            0.0480


## 15: Saving improved model

In [47]:
print("\n💾 Saving models...")

os.makedirs('../models', exist_ok=True)

# Save Random Forest (the better model)
joblib.dump(rf_model, '../models/random_forest_24h.pkl')
joblib.dump(scaler, '../models/rf_scaler.pkl')
joblib.dump(selected_features, '../models/rf_features.pkl')
joblib.dump(medians[selected_features], '../models/rf_medians.pkl')

# Save comparison metrics
comparison = {
    'test_results': {
        'naive_bayes': {
            'accuracy': float(accuracy_score(y_test, y_test_pred_nb)),
            'precision': float(precision_score(y_test, y_test_pred_nb, zero_division=0)),
            'recall': float(recall_score(y_test, y_test_pred_nb, zero_division=0)),
            'roc_auc': float(nb_test_roc)
        },
        'random_forest': {
            'accuracy': float(accuracy_score(y_test, y_test_pred_rf)),
            'precision': float(precision_score(y_test, y_test_pred_rf, zero_division=0)),
            'recall': float(recall_score(y_test, y_test_pred_rf, zero_division=0)),
            'roc_auc': float(rf_test_roc)
        }
    },
    'improvement': {
        'roc_auc_gain': float(rf_test_roc - nb_test_roc),
        'winner': 'Random Forest' if rf_test_roc > nb_test_roc else 'Naive Bayes'
    },
    'feature_importance': feature_importance.head(10).to_dict('records')
}

with open('models/model_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=4)

print("✓ Models and metrics saved")



💾 Saving models...
✓ Models and metrics saved


In [48]:
print("\n" + "="*70)
print(" "*20 + "ANALYSIS COMPLETE")
print("="*70)

print(f"\n🎯 RECOMMENDATION:")
if rf_test_roc > 0.60:
    print(f"  ✓ Random Forest shows promising results (ROC-AUC: {rf_test_roc:.4f})")
    print(f"  ✓ Consider using this for further development")
elif rf_test_roc > 0.55:
    print(f"  ⚠️  Random Forest shows slight edge (ROC-AUC: {rf_test_roc:.4f})")
    print(f"  → Try XGBoost or deep learning for better performance")
else:
    print(f"  ❌ Both models struggle with this dataset")
    print(f"  → Bitcoin price prediction is inherently difficult")
    print(f"  → Consider different features, longer timeframes, or regime detection")

print(f"\n📚 Key Insights:")
print(f"  • Naive Bayes assumes feature independence → poor for correlated technical indicators")
print(f"  • Random Forest handles feature interactions → better for financial data")
print(f"  • Class imbalance is critical → use class_weight parameter")
print(f"  • Percentile-based targets → ensures balanced classes")

print("\n" + "="*70)


                    ANALYSIS COMPLETE

🎯 RECOMMENDATION:
  ❌ Both models struggle with this dataset
  → Bitcoin price prediction is inherently difficult
  → Consider different features, longer timeframes, or regime detection

📚 Key Insights:
  • Naive Bayes assumes feature independence → poor for correlated technical indicators
  • Random Forest handles feature interactions → better for financial data
  • Class imbalance is critical → use class_weight parameter
  • Percentile-based targets → ensures balanced classes

